# P10.6-AI — Notebook 64: evaluación final subarticular

Evalúa el checkpoint congelado del Notebook 63 sobre el `internal_test` sellado. No entrena, no ajusta hiperparámetros, thresholds ni checkpoint.

**CPU:** `1 → 2 → 3 → 4A`  
**GPU:** cambiar a L4/T4 y ejecutar `1 → 2 → 3 → 4B → 5 → 6`

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `autonomousDiagnosis=false` · `officialTestAccessed=false`


In [1]:
# 1) Dependencias, Drive y rama
from __future__ import annotations
import getpass, importlib.util, json, subprocess, sys
from pathlib import Path
import torch
from google.colab import drive  # type: ignore

packages = {
    "pydicom": "pydicom",
    "timm": "timm",
    "sklearn": "scikit-learn",
    "kaggle": "kaggle",
}
missing = [pkg for mod,pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])

drive.mount("/content/drive", force_remount=False)
REPO_REF = "enzo/p10-6-ai-rsna-findings"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
if not (REPO_ROOT/".git").exists():
    subprocess.check_call(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO_ROOT)])
else:
    subprocess.check_call(["git","fetch","origin",REPO_REF], cwd=REPO_ROOT)
    subprocess.check_call(["git","checkout",REPO_REF], cwd=REPO_ROOT)
    subprocess.check_call(["git","pull","--ff-only","origin",REPO_REF], cwd=REPO_ROOT)
EVALUATION_REPO_SHA = subprocess.check_output(
    ["git","rev-parse","HEAD"], cwd=REPO_ROOT, text=True
).strip()
sys.path.insert(0, str(REPO_ROOT/"ai_service"))
from pfi_ai_service.training.rsna_subarticular_internal_test import (
    prepare_context,
    open_or_reuse_internal,
    build_cache_archive,
    localize_cache,
    evaluate_frozen,
    final_gate,
)
print({
    "device":"cuda" if torch.cuda.is_available() else "cpu",
    "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "evaluationRepoSha":EVALUATION_REPO_SHA,
    "installedNow":missing,
})


Mounted at /content/drive
{'device': 'cuda', 'gpu': 'NVIDIA L4', 'evaluationRepoSha': 'd87c63609b6de1f958d40e7fe60f60e63f8fb8b5', 'installedNow': ['pydicom']}


In [2]:
# 2) Preflight: Notebook 63, hashes, checkpoint y sello
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
context = prepare_context(PFI_ROOT, EVALUATION_REPO_SHA)
print(json.dumps({
    "status":"READY_FOR_AUTHORIZED_INTERNAL_TEST_OPEN",
    "bestEpoch":context["training"]["bestEpoch"],
    "checkpointSha256":context["actual_checkpoint_sha"],
    "trainingRepoSha":context["training"]["repoSha"],
    "evaluationRepoSha":context["evaluation_repo_sha"],
    "internalTestParsed":False,
    "officialTestAccessed":False,
}, indent=2, ensure_ascii=False))


{
  "status": "READY_FOR_AUTHORIZED_INTERNAL_TEST_OPEN",
  "bestEpoch": 6,
  "checkpointSha256": "d41262d57b13c146a48ab15f5e183cc6a55fc92724b7d0c286cea1f2ce26e84a",
  "trainingRepoSha": "62c40f528ff2349fea4c22301972547457683bed",
  "evaluationRepoSha": "d87c63609b6de1f958d40e7fe60f60e63f8fb8b5",
  "internalTestParsed": false,
  "officialTestAccessed": false
}


## 3 — Apertura autorizada

La primera ejecución lee el CSV sellado una sola vez y crea una copia de trabajo auditada. Las ejecuciones posteriores reutilizan esa copia.

In [3]:
# 3) Apertura autorizada única o reutilización
internal_manifest = open_or_reuse_internal(context)
open_record = json.loads(context["open_record"].read_text(encoding="utf-8"))
print(json.dumps({
    "status":"INTERNAL_TEST_PREPARED",
    "rows":len(internal_manifest),
    "studies":internal_manifest["study_id"].nunique(),
    "classCounts":internal_manifest["severity"].value_counts().sort_index().to_dict(),
    "authorizedSourceReadCount":open_record["authorizedSourceReadCount"],
    "officialTestAccessed":False,
}, indent=2, ensure_ascii=False))


{
  "status": "INTERNAL_TEST_PREPARED",
  "rows": 2876,
  "studies": 296,
  "classCounts": {
    "moderate": 563,
    "normal_mild": 2052,
    "severe": 261
  },
  "authorizedSourceReadCount": 1,
  "officialTestAccessed": false
}


## 4A — CPU

Construye/reutiliza el caché del internal test y crea un TAR persistente. Si las series necesarias no están completas en Drive, solicita el token de Kaggle y descarga únicamente el subconjunto requerido.

In [5]:
# 4A) Caché persistente y TAR
try:
    cache_report = build_cache_archive(
        context,
        internal_manifest,
    )
except RuntimeError as error:
    missing_data_message = (
        "No se encontró el root DICOM completo "
        "del internal test."
    )
    if missing_data_message not in str(error):
        raise

    from pfi_ai_service.training.rsna_subarticular_training import (
        download_selected_series,
    )

    print(
        "Las series DICOM del internal test no están "
        "completas en Drive. Se descargará únicamente "
        "el subconjunto requerido desde Kaggle."
    )
    kaggle_token = getpass.getpass(
        "Kaggle API token: "
    )
    download_selected_series(
        internal_manifest,
        internal_manifest,
        Path("/content/RSNA_LUMBAR_DISC"),
        "rsna-2024-lumbar-spine-degenerative-classification",
        kaggle_token,
    )
    cache_report = build_cache_archive(
        context,
        internal_manifest,
    )

print(json.dumps(
    cache_report,
    indent=2,
    ensure_ascii=False,
))


cache internal_test por serie:   0%|          | 0/349 [00:00<?, ?it/s]

{'archived': 500, 'total': 2876}
{'archived': 1000, 'total': 2876}
{'archived': 1500, 'total': 2876}
{'archived': 2000, 'total': 2876}
{'archived': 2500, 'total': 2876}
{'archived': 2876, 'total': 2876}
{
  "schemaVersion": "pfi.rsna-subarticular-internal-cache.v1",
  "ticket": "P10.6-AI",
  "notebook": 64,
  "split": "internal_test",
  "cacheRoot": "/content/drive/MyDrive/PFI_MVP/cache/notebook64_subarticular_internal_test_cache",
  "archive": "/content/drive/MyDrive/PFI_MVP/cache/notebook64_subarticular_internal_test_cache.tar",
  "archiveSha256": "edc51bfca35c8f0ea67917ccad30542fe66787882c859e822872522d128ef57a",
  "archiveGiB": 0.409,
  "expectedSamples": 2876,
  "cacheFiles": 2876,
  "cacheComplete": true,
  "cacheAudit": {
    "split": "internal_test",
    "expectedSamples": 2876,
    "builtSamples": 2876,
    "reusedSamples": 0,
    "cacheFiles": 2876,
    "minutes": 31.47
  },
  "dataRoot": "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC",
  "dataAudits": [
    {
      "r

## 4B — GPU

Después de 4A, cambiar a L4/T4 y ejecutar nuevamente `1 → 2 → 3 → 4B`.

In [4]:
# 4B) Copiar TAR al SSD local
internal_samples, CACHE_ROOT = localize_cache(context, internal_manifest)
print({
    "gpu":torch.cuda.get_device_name(0),
    "cacheRoot":str(CACHE_ROOT),
    "internalTestFiles":len(internal_samples),
    "readyForFrozenInference":True,
})


{'gpu': 'NVIDIA L4', 'cacheRoot': '/content/notebook64_subarticular_internal_test_cache', 'internalTestFiles': 2876, 'readyForFrozenInference': True}


## 5 — Evaluación final

Ejecuta inferencia con el checkpoint congelado. Si ya existe una evaluación final, se detiene.

In [5]:
# 5) Inferencia congelada, métricas y exportación
evaluation_summary = evaluate_frozen(context, internal_samples, CACHE_ROOT)
print(json.dumps(evaluation_summary, indent=2, ensure_ascii=False))


{'batch': 50, 'totalBatches': 90}
{'batch': 90, 'totalBatches': 90}
{
  "schemaVersion": "pfi.rsna-subarticular-internal-test-evaluation.v1",
  "ticket": "P10.6-AI",
  "notebook": 64,
  "sourceNotebook": 63,
  "status": "INTERNAL_TEST_EVALUATED_FOR_RESEARCH_EXPORT",
  "evaluationCompleted": true,
  "createdAtUtc": "2026-08-06T02:38:11.019712+00:00",
  "task": "subarticular_stenosis_left_right",
  "sequence": "Axial T2",
  "data": {
    "rows": 2876,
    "studies": 296,
    "classCounts": {
      "moderate": 563,
      "normal_mild": 2052,
      "severe": 261
    },
    "authorizedSourceReadCount": 1,
    "sourceManifestSha256": "109c815c8836e3f5e1d5f3be4ff284636dcd496cd08fe21d0d7f9ddb90706a6e",
    "preparedManifestSha256": "a6a1db797eba60cc3ceb3e90b987adffdb712bfddc252750545f70a0b3c31c23"
  },
  "checkpoint": {
    "path": "/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings/subarticular_axial_t2_2p5d/final_internal_test_evaluation/frozen_subarticular_checkpoint.pt",
    "sha256

In [6]:
# 6) Gate final
close_report = final_gate(context)
print(json.dumps(close_report, indent=2, ensure_ascii=False))


{
  "status": "INTERNAL_TEST_EVALUATED_FOR_RESEARCH_EXPORT",
  "metrics": {
    "balanced_accuracy": 0.6565011404151501,
    "loss": 0.6884503357931039,
    "macro_f1": 0.6284558720836352,
    "moderate_recall": 0.44582593250444047,
    "normal_mild_recall": 0.8455165692007798,
    "selection_score": 0.629630457072743,
    "severe_recall": 0.6781609195402298,
    "support": 2876,
    "weighted_log_loss": 0.6884576824836782
  },
  "checks": {
    "status": true,
    "completed": true,
    "singleSourceRead": true,
    "checkpointFrozen": true,
    "checkpointHash": true,
    "metricGatesNotApplied": true,
    "noAcceptanceDecision": true,
    "noTraining": true,
    "noHyperparameterAdjustment": true,
    "noThresholdAdjustment": true,
    "noCheckpointReselection": true,
    "humanReviewRequired": true,
    "notClinicalDiagnosis": true,
    "officialTestNotAccessed": true
  }
}


## Resultado esperado

`INTERNAL_TEST_EVALUATED_FOR_RESEARCH_EXPORT`

Las métricas se reportan sin usarlas para reentrenar o cambiar el modelo. El siguiente paso es integrar el checkpoint congelado en el runtime, manteniendo revisión humana.